In [ ]:
import pandas as pd
from tdc.multi_pred import DTI
import os

# Ensure the splits folder exists in your data directory
os.makedirs('../data/splits', exist_ok=True)

def process_dataset(dataset_name):
    print(f"========================================")
    print(f"LOADING {dataset_name} DATASET")
    print(f"========================================")
    
    # Load Data
    data = DTI(name=dataset_name)
    df = data.get_data()
    
    # Deliverable 1: Basic Stats
    num_drugs = df['Drug'].nunique()
    num_targets = df['Target'].nunique()
    num_pairs = len(df)
    min_affinity = df['Y'].min()
    max_affinity = df['Y'].max()
    
    print(f"Total Pairs: {num_pairs}")
    print(f"Unique Drugs: {num_drugs}")
    print(f"Unique Proteins: {num_targets}")
    print(f"Affinity Range: {min_affinity:.4f} to {max_affinity:.4f}\n")
    
    # Deliverable 2: Build the four splits
    print(f"Generating splits for {dataset_name} (This may take a moment)...")
    splits = {
        'random': data.get_split(),
        'cold_drug': data.get_split(method='cold_split', column_name='Drug'),
        'cold_target': data.get_split(method='cold_split', column_name='Target'),
        'cold_pair': data.get_split(method='cold_split', column_name=['Drug', 'Target'])
    }
    
    # Save all splits to the data/splits folder
    for split_name, split_dict in splits.items():
        for fold in ['train', 'valid', 'test']:
            file_path = f"../data/splits/{dataset_name.lower()}_{split_name}_{fold}.csv"
            split_dict[fold].to_csv(file_path, index=False)
            
    print(f"All 12 CSV files saved for {dataset_name}.\n")
    
    # Deliverable 2: Sanity Checks (Crucial Step)
    print(f"Running Sanity Checks for {dataset_name}...")
    for split_name in ['cold_drug', 'cold_target', 'cold_pair']:
        train_df = splits[split_name]['train']
        test_df = splits[split_name]['test']
        
        if 'drug' in split_name or 'pair' in split_name:
            overlap = set(train_df['Drug']).intersection(set(test_df['Drug']))
            print(f"[{split_name}] Drug overlap between train/test: {len(overlap)}")
            if len(overlap) > 0: print(">>> WARNING: DRUG LEAKAGE DETECTED <<<")
                
        if 'target' in split_name or 'pair' in split_name:
            overlap = set(train_df['Target']).intersection(set(test_df['Target']))
            print(f"[{split_name}] Target overlap between train/test: {len(overlap)}")
            if len(overlap) > 0: print(">>> WARNING: TARGET LEAKAGE DETECTED <<<")
    print("\n")

# Run the process for both datasets
process_dataset('DAVIS')
process_dataset('KIBA')